In [1]:
!pip install pandas networkx


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import pandas as pd
import networkx as nx
from itertools import combinations
from collections import Counter, defaultdict
import math
from fog.graph import floatsam_sparsification

In [3]:
# Charger ton CSV
csv_path = "../data/processed/films_acteurs_period:15_minfilm3.csv"
df = pd.read_csv(csv_path)

In [4]:
# Nettoyage léger
df["startYear"] = pd.to_numeric(df["startYear"], errors="coerce")
df["ordering"] = pd.to_numeric(df["ordering"], errors="coerce")

# Sécurité : enlever les lignes sans acteur ou film
df = df.dropna(subset=["tconst", "nconst", "primaryName"])

In [ ]:
# Années disponibles
all_years = sorted(df["startYear"].dropna().astype(int).unique())

In [ ]:
def most_common_value(values):
    values = [str(v).strip() for v in values if pd.notna(v) and str(v).strip() != ""]
    if not values:
        return ""
    return Counter(values).most_common(1)[0][0]

def most_common_split_value(series, sep=","):
    values = (
        series.dropna()
        .astype(str)
        .str.split(sep)
        .explode()
        .str.strip()
    )
    values = values[values != ""].tolist()
    if not values:
        return ""
    return Counter(values).most_common(1)[0][0]

def join_unique(values):
    values = [str(v).strip() for v in values if pd.notna(v) and str(v).strip() != ""]
    return "|".join(sorted(set(values)))


Nombre de noeuds : 73070
Nombre d'arêtes : 1318682


In [ ]:
# Création du graphe non orienté
G = nx.Graph()

In [ ]:
# =========================
# NOEUDS = ACTEURS
# =========================

for nconst, group in df.groupby("nconst"):
    actor_name = group["primaryName"].dropna().iloc[0] if group["primaryName"].notna().any() else ""

    movies = group["tconst"].dropna().astype(str).unique().tolist()
    titles = group["title"].dropna().astype(str).unique().tolist()
    years = sorted(group["startYear"].dropna().astype(int).unique().tolist())

    genres_list = (
        group["genres"]
        .dropna()
        .astype(str)
        .str.split(",")
        .explode()
        .str.strip()
    )
    genres_list = genres_list[genres_list != ""].tolist()

    regions = group["region"].dropna().astype(str).unique().tolist()
    languages = group["language"].dropna().astype(str).unique().tolist()
    categories = group["category"].dropna().astype(str).unique().tolist()

    attrs = {
        "label": actor_name,
        "primaryName": actor_name,

        "birthYear": int(group["birthYear"].dropna().iloc[0]) if group["birthYear"].notna().any() else "",
        "deathYear": int(group["deathYear"].dropna().iloc[0]) if group["deathYear"].notna().any() else "",

        "nb_movies": len(movies),
        "movies": "|".join(movies),
        "titles": "|".join(titles),

        "years": "|".join(map(str, years)),
        "firstYear": min(years) if years else "",
        "lastYear": max(years) if years else "",

        "genres": "|".join(sorted(set(genres_list))),
        "main_genre": Counter(genres_list).most_common(1)[0][0] if genres_list else "",

        "regions": "|".join(sorted(set(regions))),
        "main_region": most_common_value(regions),

        "languages": "|".join(sorted(set(languages))),
        "main_language": most_common_value(languages),

        "categories": "|".join(sorted(set(categories))),
        "avg_ordering": float(group["ordering"].mean()) if group["ordering"].notna().any() else "",
        "min_ordering": int(group["ordering"].min()) if group["ordering"].notna().any() else "",
        "main_roles_count": int((group["ordering"] <= 5).sum())
    }

    # Colonnes temporelles pour Gephi
    movies_by_year = group.groupby("startYear")["tconst"].nunique()

    for year in all_years:
        count = int(movies_by_year.get(year, 0))
        attrs[f"movies_{year}"] = count
        attrs[f"active_{year}"] = int(count > 0)

    G.add_node(str(nconst), **attrs)


In [ ]:
# =========================
# ARÊTES = CO-APPARITIONS
# =========================

edge_data = defaultdict(lambda: {
    "common_movies": [],
    "common_titles": [],
    "years": [],
    "genres": [],
    "regions": [],
    "languages": [],
    "ordering_sum": []
})

for tconst, group in df.groupby("tconst"):
    actors = group["nconst"].dropna().astype(str).unique().tolist()

    if len(actors) < 2:
        continue

    title = group["title"].dropna().astype(str).iloc[0] if group["title"].notna().any() else ""
    year = int(group["startYear"].dropna().iloc[0]) if group["startYear"].notna().any() else ""

    genres = (
        group["genres"]
        .dropna()
        .astype(str)
        .str.split(",")
        .explode()
        .str.strip()
    )
    genres = genres[genres != ""].unique().tolist()

    regions = group["region"].dropna().astype(str).unique().tolist()
    languages = group["language"].dropna().astype(str).unique().tolist()

    ordering_by_actor = dict(zip(group["nconst"].astype(str), group["ordering"]))

    for actor1, actor2 in combinations(sorted(actors), 2):
        key = tuple(sorted([actor1, actor2]))

        edge_data[key]["common_movies"].append(str(tconst))
        edge_data[key]["common_titles"].append(title)

        if year != "":
            edge_data[key]["years"].append(year)

        edge_data[key]["genres"].extend(genres)
        edge_data[key]["regions"].extend(regions)
        edge_data[key]["languages"].extend(languages)

        o1 = ordering_by_actor.get(actor1)
        o2 = ordering_by_actor.get(actor2)

        if pd.notna(o1) and pd.notna(o2):
            edge_data[key]["ordering_sum"].append(float(o1 + o2))

# Ajout des arêtes
for (actor1, actor2), data in edge_data.items():
    years = sorted(set(data["years"]))

    ordering_values = data["ordering_sum"]

    G.add_edge(
        actor1,
        actor2,

        weight=len(data["common_movies"]),
        common_movies_count=len(data["common_movies"]),

        common_movies="|".join(data["common_movies"]),
        common_titles="|".join(data["common_titles"]),

        years="|".join(map(str, years)),
        first_common_year=min(years) if years else "",
        last_common_year=max(years) if years else "",

        genres="|".join(sorted(set(data["genres"]))),
        main_common_genre=most_common_value(data["genres"]),

        regions="|".join(sorted(set(data["regions"]))),
        main_common_region=most_common_value(data["regions"]),

        languages="|".join(sorted(set(data["languages"]))),
        main_common_language=most_common_value(data["languages"]),

        avg_pair_ordering_sum=sum(ordering_values) / len(ordering_values) if ordering_values else ""
    )

print("Nombre de noeuds :", G.number_of_nodes())
print("Nombre d'arêtes :", G.number_of_edges())

In [6]:
def clean_value_for_gephi(x):
    """
    Nettoie les valeurs pour export GEXF/CSV Gephi.
    """
    if x is None:
        return ""

    if isinstance(x, float) and math.isnan(x):
        return ""

    if isinstance(x, (list, tuple, set)):
        return "|".join(map(str, x))

    if isinstance(x, dict):
        return str(x)

    return x


def clean_graph_for_gephi(G):
    """
    Crée une copie du graphe sans valeurs None / NaN.
    """
    H = nx.Graph()

    for node, attrs in G.nodes(data=True):
        clean_attrs = {
            key: clean_value_for_gephi(value)
            for key, value in attrs.items()
        }
        H.add_node(str(node), **clean_attrs)

    for source, target, attrs in G.edges(data=True):
        clean_attrs = {
            key: clean_value_for_gephi(value)
            for key, value in attrs.items()
        }
        H.add_edge(str(source), str(target), **clean_attrs)

    return H


G_gephi = clean_graph_for_gephi(G)

print("Noeuds :", G_gephi.number_of_nodes())
print("Arêtes :", G_gephi.number_of_edges())

# Export GEXF
gexf_path = "../data/graph/graphe_acteur_acteur.gexf"
nx.write_gexf(G_gephi, gexf_path)

print("GEXF exporté :", gexf_path)

# Export CSV noeuds
nodes_df = pd.DataFrame([
    {"Id": node, **attrs}
    for node, attrs in G_gephi.nodes(data=True)
])

nodes_path = "../data/graph/gephi_nodes_acteurs.csv"
nodes_df.to_csv(nodes_path, index=False, encoding="utf-8")

print("CSV noeuds exporté :", nodes_path)

# Export CSV arêtes
edges_df = pd.DataFrame([
    {
        "Source": source,
        "Target": target,
        "Type": "Undirected",
        **attrs
    }
    for source, target, attrs in G_gephi.edges(data=True)
])

edges_path = "../data/graph/gephi_edges_acteurs.csv"
edges_df.to_csv(edges_path, index=False, encoding="utf-8")

print("CSV arêtes exporté :", edges_path)

Noeuds : 73070
Arêtes : 1318682
GEXF exporté : ../data/graph/graphe_acteur_acteur.gexf
CSV noeuds exporté : ../data/graph/gephi_nodes_acteurs.csv
CSV arêtes exporté : ../data/graph/gephi_edges_acteurs.csv


In [14]:
# =========================
# PARAMÈTRES
# =========================

MIN_MAIN_ROLES = 2     # importance dans les films
MIN_DEGREE = 10        # centralité locale
MIN_EDGE_WEIGHT = 2    # collaborations fortes
K_CORE = 3             # coeur du graphe

# =========================
# 1. FILTRE NOEUDS
# =========================

degree = dict(G.degree())

nodes_to_keep = [
    n for n, attrs in G.nodes(data=True)
    if attrs.get("main_roles_count", 0) >= MIN_MAIN_ROLES
    and degree.get(n, 0) >= MIN_DEGREE
]

G1 = G.subgraph(nodes_to_keep).copy()

print("Après filtre noeuds")
print("Noeuds :", G1.number_of_nodes())
print("Arêtes :", G1.number_of_edges())

# =========================
# 2. FILTRE ARÊTES
# =========================

edges_to_keep = [
    (u, v, attrs)
    for u, v, attrs in G1.edges(data=True)
    if attrs.get("weight", 1) >= MIN_EDGE_WEIGHT
]

G2 = nx.Graph()
G2.add_nodes_from(G1.nodes(data=True))
G2.add_edges_from(edges_to_keep)

# Supprimer isolés
G2.remove_nodes_from(list(nx.isolates(G2)))

print("Après filtre arêtes")
print("Noeuds :", G2.number_of_nodes())
print("Arêtes :", G2.number_of_edges())

# =========================
# 3. K-CORE
# =========================

G3 = nx.k_core(G2, k=K_CORE)

print("Après k-core")
print("Noeuds :", G3.number_of_nodes())
print("Arêtes :", G3.number_of_edges())

# =========================
# 4. PLUS GRANDE COMPOSANTE
# =========================

if len(G3) > 0:
    largest_cc = max(nx.connected_components(G3), key=len)
    G_final = G3.subgraph(largest_cc).copy()
else:
    G_final = G3

print("Graphe final")
print("Noeuds :", G_final.number_of_nodes())
print("Arêtes :", G_final.number_of_edges())

Après filtre noeuds
Noeuds : 56414
Arêtes : 1107050
Après filtre arêtes
Noeuds : 38672
Arêtes : 125685
Après k-core
Noeuds : 19783
Arêtes : 101999
Graphe final
Noeuds : 19130
Arêtes : 99948


In [16]:
nx.write_gexf(G_final, "../data/graph/graphe_acteur_acteur_reduit.gexf")

nodes_df = pd.DataFrame([
    {"Id": n, **attrs}
    for n, attrs in G_final.nodes(data=True)
])

edges_df = pd.DataFrame([
    {"Source": u, "Target": v, "Type": "Undirected", **attrs}
    for u, v, attrs in G_final.edges(data=True)
])

nodes_df.to_csv("../data/graph/gephi_nodes_acteurs_reduit.csv", index=False)
edges_df.to_csv("../data/graph/gephi_edges_acteurs_reduit.csv", index=False)